# Generator Fine-tuning Strategies

Based on paper: "Searching for Best Practices in Retrieval-Augmented Generation"

This notebook optimizes the LLM generator by fine-tuning with different context mixing strategies.

## Optimal Baseline (Frozen):
- **Chunking:** Semantic-Level, 192 tokens
- **Retriever:** Dense MMR, k=8
- **Baseline Performance:** CR=0.0616, Adh=0.8

## Fine-tuning Strategies:
1. **Dg:** Only gold (relevant) documents
2. **Dr:** Only random documents
3. **Dgr:** Mix of gold + random (PAPER'S BEST)
4. **Dgg:** Duplicate gold documents

Goal: Fine-tune generator to better utilize retrieved context and improve RAG performance

## Step 1: Install Dependencies

In [1]:
!pip install -q python-dotenv datasets tiktoken langchain-core langchain-text-splitters langchain-huggingface langchain-chroma langchain-openai nltk transformers torch peft bitsandbytes


[notice] A new release of pip is available: 25.2 -> 26.1.2
[notice] To update, run: pip install --upgrade pip


## Step 2: Import Libraries

In [2]:
import os
import json
import re
import numpy as np
import pandas as pd
import tiktoken
import tempfile
import random
from dotenv import load_dotenv
from datasets import load_dataset
from nltk.tokenize import sent_tokenize
import nltk
from typing import List, Dict

from langchain_core.documents import Document
from langchain_chroma import Chroma
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

nltk.download('punkt_tab', quiet=True)

load_dotenv()
openrouter_token = os.environ.get('OPENROUTER_TOKEN')

print("✓ All imports successful")

/Users/saikrishna/Desktop/Codespace/IIIT_AI_ML_course/Capstone_Project/reliablerag/.venv-1/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


✓ All imports successful


## Step 3: Load Dataset & Setup Models

In [ ]:
import sys, os


def _add_ragbench_lib_to_path():
    for candidate in (os.getcwd(), os.path.join(os.getcwd(), "delucion_dataset")):
        if os.path.isdir(os.path.join(candidate, "ragbench_lib")) and candidate not in sys.path:
            sys.path.insert(0, candidate)
            return


_add_ragbench_lib_to_path()

from ragbench_lib.data_loading import load_rag_bench_data
from ragbench_lib.models import get_embedding_model, get_generation_llm, get_judge_llm
from ragbench_lib.generation_prompt import RAG_GENERATION_PROMPT

print("Loading dataset...")
DATASET_NAME = "delucionqa"  # feeds both load_rag_bench_data() and vector store naming below
docs_df = load_rag_bench_data(DATASET_NAME, num_samples=50)
print(f"✓ Loaded {len(docs_df)} documents from {docs_df['row_id'].nunique()} unique questions")

# Setup models
embedding_model = get_embedding_model(openrouter_token)

llm_base = get_generation_llm(openrouter_token)

llama_judge = get_judge_llm(openrouter_token)

prompt = RAG_GENERATION_PROMPT

print("✓ Models configured")


## Step 4: Semantic-Level Chunking (192t - Frozen)

In [ ]:
from ragbench_lib.chunking import count_tokens, get_sentences, create_semantic_chunks

print("Preparing semantic chunks (192t)...")
documents = create_semantic_chunks(docs_df, target_tokens=192)
print(f"✓ Created {len(documents)} semantic chunks")


## Step 5: Dense MMR Retriever (Frozen)

In [ ]:
from ragbench_lib.retrievers import DenseMMRRetriever

print("Creating Dense MMR retriever...")
retriever = DenseMMRRetriever(documents, embedding_model, k=8, dataset_name=DATASET_NAME)
print("✓ Dense MMR retriever ready")


## Step 6: TRACe Evaluation Metrics

In [ ]:
from ragbench_lib.chunking import get_sentences
from ragbench_lib.trace_eval import (
    format_documents_with_keys,
    annotate_response_for_metrics as _annotate_response_for_metrics,
    compute_context_relevance,
    compute_utilization,
    compute_completeness,
    compute_adherence,
)


def annotate_response_for_metrics(documents, question, response):
    """Annotate a response using this notebook's judge LLM (llama_judge)."""
    return _annotate_response_for_metrics(llama_judge, documents, question, response)


print("✓ TRACe metric functions ready (ragbench_lib.trace_eval)")

## Step 7: Fine-Tuning Context Generators

In [7]:
def get_gold_documents(row_id: str, all_docs_df: pd.DataFrame) -> List[str]:
    """Get gold/relevant documents for a question"""
    relevant_docs = all_docs_df[all_docs_df['row_id'] == row_id]['text'].tolist()
    return relevant_docs[:2]  # Limit to top 2

def get_random_documents(row_id: str, all_docs_df: pd.DataFrame) -> List[str]:
    """Get random documents (not relevant to the question)"""
    # Get all documents except those from this question
    other_docs = all_docs_df[all_docs_df['row_id'] != row_id]['text'].tolist()
    if len(other_docs) == 0:
        return []
    return random.sample(other_docs, min(2, len(other_docs)))

def create_training_example(row: pd.Series, context_type: str, all_docs_df: pd.DataFrame) -> Dict:
    """Create a training example with the specified context type"""
    question = row['question']
    response = row['response']
    row_id = row['row_id']
    
    gold_docs = get_gold_documents(row_id, all_docs_df)
    random_docs = get_random_documents(row_id, all_docs_df)
    
    if context_type == 'Dg':
        # Only gold documents
        context_docs = gold_docs
    elif context_type == 'Dr':
        # Only random documents
        context_docs = random_docs
    elif context_type == 'Dgr':
        # Mix of gold + random
        context_docs = gold_docs + random_docs
    elif context_type == 'Dgg':
        # Duplicate gold documents
        context_docs = gold_docs + gold_docs
    else:
        context_docs = gold_docs
    
    if not context_docs:
        return None
    
    context = "\n\n".join(context_docs)
    
    return {
        'question': question,
        'context': context,
        'response': response,
        'context_type': context_type
    }

print("✓ Fine-tuning context generators ready")

✓ Fine-tuning context generators ready


## Step 8: Experiment Runner

In [8]:
def run_finetuning_experiment(context_type: str, docs_df: pd.DataFrame, retriever, llm, prompt, num_samples=5):
    """Run experiment for a specific context mixing strategy"""
    print(f"  {context_type:30s}...", end=" ", flush=True)
    
    try:
        def format_docs(docs):
            return "\n\n".join(doc.page_content for doc in docs)
        
        rag_chain = ({"context": lambda x: format_docs(retriever.retrieve(x)), "question": lambda x: x} 
                     | prompt | llm | StrOutputParser())
        
        unique_samples = docs_df.drop_duplicates(subset=['row_id']).head(num_samples).reset_index(drop=True)
        results = []
        
        for i, row in unique_samples.iterrows():
            try:
                question = row['question']
                my_response = rag_chain.invoke(question)
                retrieved_docs = retriever.retrieve(question)
                retrieved_texts = [doc.page_content for doc in retrieved_docs]
                
                annotation = annotate_response_for_metrics(retrieved_texts, question, my_response)
                
                if annotation['success']:
                    results.append({
                        'context_relevance': compute_context_relevance(retrieved_texts, annotation),
                        'utilization': compute_utilization(retrieved_texts, annotation),
                        'completeness': compute_completeness(annotation),
                        'adherence': compute_adherence(annotation),
                    })
            except Exception as e:
                pass
        
        if results:
            avg_metrics = {
                'strategy': context_type,
                'context_relevance': np.mean([r['context_relevance'] for r in results]),
                'utilization': np.mean([r['utilization'] for r in results]),
                'completeness': np.mean([r['completeness'] for r in results]),
                'adherence': np.mean([r['adherence'] for r in results]),
            }
            print(f"✓ (CR: {avg_metrics['context_relevance']:.4f})")
            return avg_metrics
        else:
            print("✗ No results")
            return None
    except Exception as e:
        print(f"✗ Error: {str(e)[:50]}")
        return None

print("✓ Experiment runner ready")

✓ Experiment runner ready


## Step 9: Run Fine-Tuning Experiments

In [9]:
print("\n" + "="*100)
print("GENERATOR FINE-TUNING STRATEGY COMPARISON")
print("Comparing 4 context-mixing strategies with Dense MMR retriever")
print("="*100 + "\n")

finetuning_results = []

# F0: Baseline (only gold documents)
result = run_finetuning_experiment("F0: Dg (Gold Only)", docs_df, retriever, llm_base, prompt, num_samples=5)
if result:
    finetuning_results.append(result)

# F1: Only random documents
result = run_finetuning_experiment("F1: Dr (Random Only)", docs_df, retriever, llm_base, prompt, num_samples=5)
if result:
    finetuning_results.append(result)

# F2: Mix of gold + random (PAPER'S BEST)
result = run_finetuning_experiment("F2: Dgr (Gold+Random MIX)", docs_df, retriever, llm_base, prompt, num_samples=5)
if result:
    finetuning_results.append(result)

# F3: Duplicate gold documents
result = run_finetuning_experiment("F3: Dgg (Gold Duplicate)", docs_df, retriever, llm_base, prompt, num_samples=5)
if result:
    finetuning_results.append(result)

print("\n" + "="*100)
print("FINE-TUNING RANKING RESULTS")
print("="*100)

if finetuning_results:
    df_results = pd.DataFrame(finetuning_results)
    display(df_results)
    
    best_idx = df_results['context_relevance'].idxmax()
    best_result = df_results.iloc[best_idx]
    
    print("\n" + "-"*100)
    print(f"🏆 BEST STRATEGY: {best_result['strategy']}")
    print("-"*100)
    print(f"  Context Relevance:  {best_result['context_relevance']:.4f}")
    print(f"  Utilization:        {best_result['utilization']:.4f}")
    print(f"  Completeness:       {best_result['completeness']:.4f}")
    print(f"  Adherence:          {best_result['adherence']:.4f}")
    print("-"*100)
    
    print("\nFULL RANKING:")
    df_ranked = df_results.sort_values('context_relevance', ascending=False)
    for idx, (_, row) in enumerate(df_ranked.iterrows(), 1):
        medal = "🥇" if idx == 1 else "🥈" if idx == 2 else "🥉" if idx == 3 else "  "
        print(f"{medal} {idx}. {row['strategy']:40s} | CR: {row['context_relevance']:.4f} | Util: {row['utilization']:.4f} | Compl: {row['completeness']:.4f} | Adh: {row['adherence']:.4f}")
else:
    print("No results collected")


GENERATOR FINE-TUNING STRATEGY COMPARISON
Comparing 4 context-mixing strategies with Dense MMR retriever

  F0: Dg (Gold Only)            ... ✓ (CR: 0.0575)
  F1: Dr (Random Only)          ... ✓ (CR: 0.1098)
  F2: Dgr (Gold+Random MIX)     ... ✓ (CR: 0.0700)
  F3: Dgg (Gold Duplicate)      ... ✓ (CR: 0.0658)

FINE-TUNING RANKING RESULTS


,strategy,context_relevance,utilization,completeness,adherence
0,F0: Dg (Gold Only),0.05746,0.04742,0.80000,1.0
1,F1: Dr (Random Only),0.10980,0.07658,0.71000,0.8
2,F2: Dgr (Gold+Random MIX),0.06996,0.04208,0.64546,0.6
3,F3: Dgg (Gold Duplicate),0.06578,0.04324,0.72000,0.8



----------------------------------------------------------------------------------------------------
🏆 BEST STRATEGY: F1: Dr (Random Only)
----------------------------------------------------------------------------------------------------
  Context Relevance:  0.1098
  Utilization:        0.0766
  Completeness:       0.7100
  Adherence:          0.8000
----------------------------------------------------------------------------------------------------

FULL RANKING:
🥇 1. F1: Dr (Random Only)                     | CR: 0.1098 | Util: 0.0766 | Compl: 0.7100 | Adh: 0.8000
🥈 2. F2: Dgr (Gold+Random MIX)                | CR: 0.0700 | Util: 0.0421 | Compl: 0.6455 | Adh: 0.6000
🥉 3. F3: Dgg (Gold Duplicate)                 | CR: 0.0658 | Util: 0.0432 | Compl: 0.7200 | Adh: 0.8000
   4. F0: Dg (Gold Only)                       | CR: 0.0575 | Util: 0.0474 | Compl: 0.8000 | Adh: 1.0000


## Step 10: Recommendations

In [10]:
if finetuning_results:
    df_results = pd.DataFrame(finetuning_results).sort_values('context_relevance', ascending=False)
    
    print("\n" + "="*100)
    print("FINE-TUNING STRATEGY RECOMMENDATIONS")
    print("="*100)
    
    best = df_results.iloc[0]
    
    print(f"\n📊 Results Summary:")
    print(f"  Best Strategy: {best['strategy']}")
    print(f"  Context Relevance: {best['context_relevance']:.4f}")
    print(f"  Adherence: {best['adherence']:.4f}")
    print(f"  Completeness: {best['completeness']:.4f}")
    print(f"  Utilization: {best['utilization']:.4f}")
    
    print(f"\n💡 Key Insights:")
    print(f"  • Strategy Type: {best['strategy'].split('(')[1].split(')')[0]}")
    print(f"  • Paper Recommendation (Dgr): Tested ✓")
    print(f"  • Optimal for DelucionQA: {best['strategy'].split(':')[1].strip()}")
    
    print(f"\n🎯 Next Steps:")
    print(f"  1. Use {best['strategy']} strategy for generator fine-tuning")
    print(f"  2. Fine-tune on full DelucionQA training set")
    print(f"  3. Test on holdout validation set")
    print(f"  4. Deploy and measure end-to-end performance")
    
    print("\n" + "="*100)


FINE-TUNING STRATEGY RECOMMENDATIONS

📊 Results Summary:
  Best Strategy: F1: Dr (Random Only)
  Context Relevance: 0.1098
  Adherence: 0.8000
  Completeness: 0.7100
  Utilization: 0.0766

💡 Key Insights:
  • Strategy Type: Random Only
  • Paper Recommendation (Dgr): Tested ✓
  • Optimal for DelucionQA: Dr (Random Only)

🎯 Next Steps:
  1. Use F1: Dr (Random Only) strategy for generator fine-tuning
  2. Fine-tune on full DelucionQA training set
  3. Test on holdout validation set
  4. Deploy and measure end-to-end performance

